# 预训练数据盘点与列表生成（Drive + GCS）

本笔记本用于：

1. **扫描 Google Drive** `dataset/pretrain/` 下各数据集目录（子结构不一致时递归收集 `.npy`）。
2. **检查 GCS**：与仓库内现有 ipynb 对齐的三套路径——
   - `gs://brats_preprocessed_npy`（`colab_pretrain_GCS*.ipynb`）
   - `gs://flare23`（`colab_flare23_preprocess.ipynb`）
   - `gs://rsna_2022_ped_preprocessed_npy/npy2`（`colab_rsna_ped_zip_to_gcs.ipynb`）
3. **生成训练列表**（与 `newFullPretrain/image_dataset.py` 兼容）：
   - **spatial**：单路径一行，本地 `.npy`（或后续你可自行把前缀改成 gcsfuse 挂载路径）。
   - **contrast（BraTS）**：同一 case 四个模态 **`gs://...t1n.npy,...,t2f.npy` 逗号一行**（避免单行里单独 `gs://` 被误判为 tar 格式）。
   - **semantic（DeepLesion）**：四类病灶各采样拼成一行（逗号分隔），与 `colab_pretrain_GCS.ipynb` 思路一致。

**说明**：`np.load` 不能直接读 `gs://…`；线上训练请 **gcsfuse 挂载**或先把列表里的前缀批量替换为本地挂载目录。

**TCIA**：旧笔记本输出在 `dataset/TCIA_Covid19/patch_random_spatial`，若 `pretrain/TCIA` 不存在会自动探测该路径。

In [ ]:
# Cell 1: 挂载 Google Drive
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Cell 2: 依赖（DeepLesion 语义列表需 numpy）
!pip install -q numpy tqdm

In [ ]:
# Cell 3: 路径配置与工具函数
from __future__ import annotations

import json
import os
import random
import subprocess
from collections import defaultdict
from pathlib import Path

import numpy as np
from tqdm.auto import tqdm

DRIVE_PRETRAIN = Path("/content/drive/MyDrive/dataset/pretrain")
DRIVE_DATASET = Path("/content/drive/MyDrive/dataset")

# 列表与盘点输出目录（在 Drive 上持久化）
LISTS_OUT = DRIVE_PRETRAIN / "pretrain_lists_inventory"
LISTS_OUT.mkdir(parents=True, exist_ok=True)
(LISTS_OUT / "per_dataset").mkdir(exist_ok=True)

# Drive 上期望的数据集文件夹名（与实际文件夹匹配时不区分大小写）
DRIVE_DATASET_NAMES = [
    "AbdomenCT-1K",
    "Amos",
    "DeepLesion",
    "FLARE22",
    "FLARE23",
    "ISLES",
    "RibFrac",
    "RSNA-2022-CSFD",
    "STOIC",
    "TCIA",
    "TotalSegmentator",
]

# 额外纳入 spatial 的根路径（旧 notebook 约定）
EXTRA_SPATIAL_ROOTS = [
    DRIVE_DATASET / "LIDC-IDRI" / "patch_random_spatial",
    DRIVE_DATASET / "TCIA_Covid19" / "patch_random_spatial",
]

# 现有 ipynb 中约定的三套 GCS
GCS_DATASETS = {
    "brats_preprocessed_npy": "gs://brats_preprocessed_npy",
    "flare23": "gs://flare23",
    "rsna_2022_ped_npy2": "gs://rsna_2022_ped_preprocessed_npy/npy2",
}

SKIP_DIR_NAMES = {"__MACOSX", ".git"}


def find_under_pretrain(preferred_name: str) -> Path | None:
    """在 pretrain 根下按不区分大小写匹配子文件夹。"""
    if not DRIVE_PRETRAIN.is_dir():
        return None
    exact = DRIVE_PRETRAIN / preferred_name
    if exact.is_dir():
        return exact
    lower = preferred_name.lower()
    for child in DRIVE_PRETRAIN.iterdir():
        if child.is_dir() and child.name.lower() == lower:
            return child
    return None


def npy_roots_for_dataset(ds_root: Path) -> list[Path]:
    """
    与各 Colab 预处理 notebook 对齐：优先收集 npy / npy1..4 / patch_random_spatial。
    若均不存在则退回对整个数据集根递归（适配自定义布局）。
    """
    tagged = []
    for sub in ("npy", "npy1", "npy2", "npy3", "npy4", "patch_random_spatial"):
        p = ds_root / sub
        if p.is_dir():
            tagged.append(p)
    if tagged:
        return tagged
    return [ds_root] if ds_root.is_dir() else []


def iter_npys_under(root: Path):
    if not root.is_dir():
        return
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [d for d in dirnames if d not in SKIP_DIR_NAMES]
        if "__MACOSX" in dirpath:
            continue
        for fn in filenames:
            if fn.endswith(".npy"):
                yield Path(dirpath) / fn


def collect_npys_for_roots(roots: list[Path]) -> list[str]:
    out: list[str] = []
    seen = set()
    for r in roots:
        for p in iter_npys_under(r):
            s = str(p)
            if s not in seen:
                seen.add(s)
                out.append(s)
    out.sort()
    return out


def write_lines(path: Path, lines: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))


def gsutil_ls_r_npy(gcs_prefix: str, timeout_sec: int = 7200) -> list[str]:
    """递归列出某 gs 前缀下所有 .npy（大桶可能很慢）。"""
    cmd = f"gsutil ls -r '{gcs_prefix}/**/*.npy' 2>/dev/null"
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout_sec)
    lines = [x.strip() for x in r.stdout.splitlines() if x.strip().endswith(".npy")]
    return lines


def gcs_sample_lines(lines: list[str], k: int = 15) -> list[str]:
    return lines[:k]


print("LISTS_OUT =", LISTS_OUT)

## Drive：递归扫描各数据集

对每个数据集输出：`存在与否`、`npy 数量`、每个候选根路径统计；并写入 `per_dataset/<name>_spatial.txt`（DeepLesion 单独可用于语义）。

In [ ]:
# Cell 4: Drive 扫描
inventory = {"drive_pretrain": str(DRIVE_PRETRAIN), "datasets": {}}

if not DRIVE_PRETRAIN.is_dir():
    print("未找到", DRIVE_PRETRAIN, "请先挂载 Drive 并完成预处理。")
else:
    for name in tqdm(DRIVE_DATASET_NAMES, desc="Drive datasets"):
        ds_root = find_under_pretrain(name)
        entry = {"resolved_root": str(ds_root) if ds_root else None, "npy_total": 0, "roots": []}
        if ds_root is None:
            inventory["datasets"][name] = entry
            continue
        roots = npy_roots_for_dataset(ds_root)
        all_paths: list[str] = []
        for r in roots:
            sub = collect_npys_for_roots([r])
            entry["roots"].append({"path": str(r), "count": len(sub)})
            all_paths.extend(sub)
        all_paths = sorted(set(all_paths))
        entry["npy_total"] = len(all_paths)
        inventory["datasets"][name] = entry
        out_txt = LISTS_OUT / "per_dataset" / f"{name.replace('/', '_')}_spatial.txt"
        write_lines(out_txt, all_paths)

    # 额外 spatial 根（LIDC / TCIA 旧路径）
    for extra in EXTRA_SPATIAL_ROOTS:
        key = f"EXTRA:{extra.name}@{extra.parent.name}"
        paths = collect_npys_for_roots([extra])
        inventory["datasets"][key] = {
            "resolved_root": str(extra),
            "npy_total": len(paths),
            "roots": [{"path": str(extra), "count": len(paths)}],
        }
        write_lines(
            LISTS_OUT / "per_dataset" / f"EXTRA_{extra.parent.name}_{extra.name}_spatial.txt",
            paths,
        )

    summary_json = LISTS_OUT / "drive_inventory.json"
    with open(summary_json, "w", encoding="utf-8") as f:
        json.dump(inventory, f, indent=2, ensure_ascii=False)
    print("已写入", summary_json)

    for name, ent in sorted(inventory["datasets"].items()):
        print(f"- {name}: npy={ent['npy_total']} root={ent['resolved_root']}")

## GCS：抽样列出（需 Google 账号授权）

大桶完整递归可能耗时很长；默认对每个 bucket **抽样列出前若干条**并写入 `gcs_samples/`。若需要完整列表可把下一格里的 `FULL_SCAN = True`（耗时与费用需注意）。

In [ ]:
# Cell 5: GCS 认证 + gsutil 抽样
from google.colab import auth

auth.authenticate_user()

FULL_SCAN = False  # True = 递归全量 .npy 列表（很慢）
GCS_SAMPLE_CAP = 50

gcs_out = LISTS_OUT / "gcs_samples"
gcs_out.mkdir(parents=True, exist_ok=True)

gcs_inventory = {}

for label, prefix in GCS_DATASETS.items():
    print("\n===", label, prefix, "===")
    if FULL_SCAN:
        files = gsutil_ls_r_npy(prefix.rstrip("/"))
    else:
        # 快速：`gsutil ls` 仅一层或使用 wildcard 少量预览
        quick = subprocess.run(
            f"gsutil ls '{prefix}/**' 2>/dev/null | head -n {GCS_SAMPLE_CAP}",
            shell=True,
            capture_output=True,
            text=True,
            timeout=120,
        )
        files = [x.strip() for x in quick.stdout.splitlines() if x.strip()]

    gcs_inventory[label] = {"prefix": prefix, "listed": len(files), "sample": files[:20]}
    sample_path = gcs_out / f"{label}_sample.txt"
    write_lines(sample_path, files[:GCS_SAMPLE_CAP])
    print("样本条数:", len(files), "示例路径写入", sample_path)

with open(LISTS_OUT / "gcs_inventory_partial.json", "w", encoding="utf-8") as f:
    json.dump(gcs_inventory, f, indent=2, ensure_ascii=False)

## 合并 spatial + BraTS contrast（逗号四模态）+ DeepLesion semantic

- **spatial_merged.txt**：合并 Drive 各数据集 `per_dataset/*_spatial.txt`，默认 **排除 `DeepLesion`**（放入语义）。
- **contrast_brats_gcs_pair4.txt**：基于 `gsutil` 全量 `.npy` 列表配对四个模态（需将上一节 `FULL_SCAN = True` 且 BraTS 单元成功跑完）。
- **semantic_deeplesion.txt**：自 Drive `DeepLesion/npy`。

In [ ]:
# Cell 6: 合并列表 + BraTS + DeepLesion
random.seed(42)

# ---------- 6a 合并 spatial ----------
spatial_merge: list[str] = []
seen_sp = set()

per_dir = LISTS_OUT / "per_dataset"
if per_dir.is_dir():
    for fp in sorted(per_dir.glob("*_spatial.txt")):
        if fp.name.startswith("DeepLesion"):
            continue
        for line in open(fp, encoding="utf-8"):
            s = line.strip()
            if not s or s in seen_sp:
                continue
            seen_sp.add(s)
            spatial_merge.append(s)

spatial_merge.sort()
write_lines(LISTS_OUT / "spatial_merged_drive.txt", spatial_merge)
print("spatial_merged_drive.txt lines:", len(spatial_merge))

# ---------- 6b DeepLesion semantic（4-in-1） ----------
deeplesion_root = find_under_pretrain("DeepLesion")
dl_npy = deeplesion_root / "npy" if deeplesion_root else None
semantic_lines: list[str] = []
if dl_npy and dl_npy.is_dir():
    grouped = defaultdict(list)
    for fn in tqdm(os.listdir(dl_npy), desc="DeepLesion group"):
        if not fn.endswith(".npy"):
            continue
        full_path = str(dl_npy / fn)
        try:
            data = np.load(full_path, mmap_mode="r")
            if data.size == 0:
                continue
        except Exception:
            continue
        stem = fn.replace(".npy", "")
        lesion_type = stem.split("_")[-1] if "_" in stem else "0"
        grouped[lesion_type].append(full_path)
    for _, paths in grouped.items():
        if len(paths) < 4:
            continue
        for _ in range(len(paths)):
            semantic_lines.append(",".join(random.sample(paths, 4)))
    write_lines(LISTS_OUT / "semantic_deeplesion.txt", semantic_lines)
    print("semantic_deeplesion.txt lines:", len(semantic_lines), "types:", len(grouped))
else:
    write_lines(LISTS_OUT / "semantic_deeplesion.txt", [])
    print("DeepLesion/npy 不存在，已写空 semantic_deeplesion.txt")

# ---------- 6c BraTS GCS pair4（可选：依赖 FULL_SCAN + gsutil 全量列表） ----------
BRATS_GCS = GCS_DATASETS["brats_preprocessed_npy"]
tmp_brats = Path("/tmp/all_brats_gcs_npy.txt")
print("\n如需生成 contrast_brats_gcs_pair4.txt，请在上一节将 FULL_SCAN=True，并在下方取消注释手动运行 gsutil 转储；")
print("或在本机 shell 运行：")
print(f"  gsutil ls -r '{BRATS_GCS}/**/*.npy' > {{tmp_brats}}")

if tmp_brats.is_file() and tmp_brats.stat().st_size > 0:
    all_files = [x.strip() for x in open(tmp_brats, encoding="utf-8") if x.strip().endswith(".npy")]
    all_set = set(all_files)
    t1n_files = [f for f in all_files if ".t1n.npy" in f]
    pair4: list[str] = []
    for t1n in tqdm(t1n_files, desc="BraTS pair4"):
        base = t1n.replace(".t1n.npy", "")
        req = [f"{base}.t1n.npy", f"{base}.t1c.npy", f"{base}.t2w.npy", f"{base}.t2f.npy"]
        if all(x in all_set for x in req):
            pair4.append(",".join(req))
    write_lines(LISTS_OUT / "contrast_brats_gcs_pair4.txt", pair4)
    print("contrast_brats_gcs_pair4.txt lines:", len(pair4))
else:
    write_lines(LISTS_OUT / "contrast_brats_gcs_pair4.txt", [])
    print("未检测到", tmp_brats, "— 跳过 BraTS pair4（可先 gsutil 导出后再重新运行本单元）。")

# ---------- 6d RSNA PED：若已有全量 gs 列表文件，可追加到 spatial ----------
tmp_rsna = Path("/tmp/all_rsna_ped_npy.txt")
if tmp_rsna.is_file() and tmp_rsna.stat().st_size > 0:
    rsna_paths = [x.strip() for x in open(tmp_rsna, encoding="utf-8") if x.strip().endswith(".npy")]
    write_lines(LISTS_OUT / "spatial_rsna_ped_gcs.txt", sorted(rsna_paths))
    print("spatial_rsna_ped_gcs.txt lines:", len(rsna_paths))

print("\n完成。请将 newFullPretrain/configs/datasets.py 中 spatial_path / contrast_path / semantic_path 指向：")
print(" ", LISTS_OUT / "spatial_merged_drive.txt")
print(" ", LISTS_OUT / "contrast_brats_gcs_pair4.txt")
print(" ", LISTS_OUT / "semantic_deeplesion.txt")